# 17 Localization bootstrap confidence intervals

Computes nonparametric bootstrap confidence intervals for pathological-only hotspot localization metrics produced by notebook 12.

In [5]:

from pathlib import Path
import os, json, glob, shutil, warnings, math, random
from datetime import datetime, timezone
import numpy as np
import pandas as pd

SEED = int(os.environ.get("THERMO_SEED", "42"))
random.seed(SEED)
np.random.seed(SEED)

_candidate_bases = [Path(os.environ.get("THERMO_BASE_DIR", "")) if os.environ.get("THERMO_BASE_DIR") else None,
                    Path("/content"), Path("/mnt/data"), Path.cwd(), Path("/tmp")]
_candidate_bases = [p for p in _candidate_bases if p is not None]

def _base_is_usable(p):
    try:
        return p.exists() and os.access(p, os.W_OK)
    except Exception:
        return False

BASE_DIR = next((p for p in _candidate_bases if _base_is_usable(p)), Path("/tmp"))
PROJECT_NAME = os.environ.get("THERMO_PROJECT_NAME", "project_thermography_equine")
PROJECT_ROOT = BASE_DIR / PROJECT_NAME
DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
CONFIG_DIR = OUTPUT_ROOT / "config"
REPORTS_DIR = OUTPUT_ROOT / "reports"
TABLES_DIR = OUTPUT_ROOT / "tables"
FIGURES_DIR = OUTPUT_ROOT / "figures"
MODELS_DIR = OUTPUT_ROOT / "models"
GRADCAM_DIR = OUTPUT_ROOT / "gradcam"
XAI_DIR = OUTPUT_ROOT / "xai_integrated_gradients"
for d in [PROJECT_ROOT, DATA_ROOT, OUTPUT_ROOT, CONFIG_DIR, REPORTS_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR, GRADCAM_DIR, XAI_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEARCH_ROOTS = [PROJECT_ROOT, OUTPUT_ROOT, CONFIG_DIR, REPORTS_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR, GRADCAM_DIR, BASE_DIR, Path('/content'), Path('/mnt/data'), Path.cwd()]

def find_first(patterns, roots=SEARCH_ROOTS, required=False):
    hits=[]
    for root in roots:
        root=Path(root)
        if not root.exists():
            continue
        for pat in patterns:
            hits.extend([p for p in root.rglob(pat) if p.is_file() and '.ipynb_checkpoints' not in p.parts])
    hits=sorted(set(hits), key=lambda p: p.stat().st_mtime, reverse=True)
    if required and not hits:
        raise FileNotFoundError(f"Could not find any of: {patterns}")
    return hits[0] if hits else None

def read_csv_found(patterns, required_cols=None, required=True):
    path=find_first(patterns, required=required)
    if path is None:
        return None, None
    df=pd.read_csv(path)
    if required_cols:
        missing=[c for c in required_cols if c not in df.columns]
        if missing:
            raise KeyError(f"{path} missing required columns: {missing}")
    print(f"Loaded {path} shape={df.shape}")
    return df, path

def save_text(path, text):
    path=Path(path); path.parent.mkdir(parents=True, exist_ok=True); path.write_text(text, encoding='utf-8')

print('BASE_DIR:', BASE_DIR)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)

from sklearn.utils import resample

RNG = np.random.default_rng(SEED)
N_BOOT = int(os.environ.get("THERMO_BOOTSTRAP_N", "5000"))

pointing, pointing_path = read_csv_found(["pointing_game_results*.csv"],
    required_cols=["image_name", "strict_pointing_hit", "cam_cell_bbox_overlap", "cam_cell_bbox_iou", "distance_px", "gradcam_max_on_border"], required=True)
# Optional thresholded heatmap IoU results may contain failed rows; use only numeric rows.
iou, iou_path = read_csv_found(["iou_results*.csv"], required=False)

# Coerce types.
for c in ["strict_pointing_hit", "cam_cell_bbox_overlap", "gradcam_max_on_border"]:
    pointing[c] = pointing[c].astype(str).str.lower().isin(["true", "1", "yes"])
for c in ["cam_cell_bbox_iou", "distance_px"]:
    pointing[c] = pd.to_numeric(pointing[c], errors='coerce')

metrics = {
    "strict_pointing_accuracy": lambda d: d["strict_pointing_hit"].mean(),
    "cam_cell_overlap_accuracy": lambda d: d["cam_cell_bbox_overlap"].mean(),
    "median_cam_cell_bbox_iou": lambda d: d["cam_cell_bbox_iou"].median(),
    "mean_cam_cell_bbox_iou": lambda d: d["cam_cell_bbox_iou"].mean(),
    "median_distance_px": lambda d: d["distance_px"].median(),
    "mean_distance_px": lambda d: d["distance_px"].mean(),
    "border_max_fraction": lambda d: d["gradcam_max_on_border"].mean(),
}
rows=[]
for name, fn in metrics.items():
    point=float(fn(pointing))
    vals=[]
    n=len(pointing)
    for _ in range(N_BOOT):
        idx=RNG.integers(0,n,n)
        vals.append(float(fn(pointing.iloc[idx])))
    vals=np.asarray(vals, dtype=float)
    rows.append({
        "analysis_cohort": "pathological_primary_localization",
        "metric": name,
        "point_estimate": point,
        "bootstrap_mean": float(np.nanmean(vals)),
        "ci_lower": float(np.nanpercentile(vals, 2.5)),
        "ci_upper": float(np.nanpercentile(vals, 97.5)),
        "n_bootstrap": N_BOOT,
        "n_rows": int(n),
        "n_images": int(pointing['image_name'].nunique()),
    })

if iou is not None and not iou.empty:
    if "cam_bbox_iou" in iou.columns:
        iou["cam_bbox_iou"] = pd.to_numeric(iou["cam_bbox_iou"], errors='coerce')
        for q, sub in iou.groupby("heatmap_quantile") if "heatmap_quantile" in iou.columns else [(np.nan, iou)]:
            sub=sub.dropna(subset=["cam_bbox_iou"])
            if len(sub) == 0:
                continue
            for metric_name, fn in {
                "median_thresholded_heatmap_to_box_iou": lambda d: d["cam_bbox_iou"].median(),
                "mean_thresholded_heatmap_to_box_iou": lambda d: d["cam_bbox_iou"].mean(),
                "any_thresholded_heatmap_box_overlap_fraction": lambda d: d["cam_bbox_any_overlap"].astype(str).str.lower().isin(["true","1","yes"]).mean() if "cam_bbox_any_overlap" in d.columns else np.nan,
            }.items():
                vals=[]; n=len(sub)
                point=float(fn(sub))
                for _ in range(N_BOOT):
                    idx=RNG.integers(0,n,n)
                    vals.append(float(fn(sub.iloc[idx])))
                vals=np.asarray(vals, dtype=float)
                rows.append({
                    "analysis_cohort": f"pathological_primary_localization_q{q}",
                    "metric": metric_name,
                    "point_estimate": point,
                    "bootstrap_mean": float(np.nanmean(vals)),
                    "ci_lower": float(np.nanpercentile(vals, 2.5)),
                    "ci_upper": float(np.nanpercentile(vals, 97.5)),
                    "n_bootstrap": N_BOOT,
                    "n_rows": int(n),
                    "n_images": int(sub['image_name'].nunique()) if 'image_name' in sub.columns else np.nan,
                })

ci = pd.DataFrame(rows)
for p in [CONFIG_DIR/"localization_bootstrap_confidence_intervals.csv", REPORTS_DIR/"localization_bootstrap_confidence_intervals.csv", TABLES_DIR/"table_localization_bootstrap_confidence_intervals.csv"]:
    ci.to_csv(p, index=False)

diagnostics={
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "analysis": "localization_bootstrap_confidence_intervals",
    "pointing_input": str(pointing_path),
    "iou_input": str(iou_path) if iou_path else None,
    "n_bootstrap": N_BOOT,
    "n_rows": int(len(pointing)),
    "n_images": int(pointing['image_name'].nunique()),
    "bootstrap_level": "hotspot_annotation_row",
    "primary_endpoint": "pathological-only hotspot localization",
}
save_text(CONFIG_DIR/"localization_bootstrap_ci_diagnostics.json", json.dumps(diagnostics, indent=2))
save_text(REPORTS_DIR/"methods_localization_bootstrap_ci_text.txt", "Uncertainty around hotspot-localization metrics was estimated by nonparametric bootstrap resampling of pathological hotspot-annotation rows. For each bootstrap sample, strict pointing-game accuracy, CAM-cell overlap accuracy, CAM-cell IoU, distance-based metrics, and border-activation fraction were recomputed. Percentile 95% confidence intervals were reported.\n")
print(ci.to_string(index=False))


BASE_DIR: /content
PROJECT_ROOT: /content/project_thermography_equine
OUTPUT_ROOT: /content/project_thermography_equine/outputs
Loaded /content/project_thermography_equine/outputs/config/pointing_game_results.csv shape=(26, 6)
Loaded /content/project_thermography_equine/outputs/config/iou_results.csv shape=(52, 8)
                  analysis_cohort                    metric  point_estimate  bootstrap_mean  ci_lower  ci_upper  n_bootstrap  n_rows  n_images
pathological_primary_localization  strict_pointing_accuracy        0.038462        0.037600  0.000000  0.115385         5000      26        13
pathological_primary_localization cam_cell_overlap_accuracy        0.384615        0.382362  0.192308  0.576923         5000      26        13
pathological_primary_localization  median_cam_cell_bbox_iou        0.000000        0.002903  0.000000  0.032079         5000      26        13
pathological_primary_localization    mean_cam_cell_bbox_iou        0.034740        0.034660  0.015340  0.057225 

## Completion
Use `table_localization_bootstrap_confidence_intervals.csv` in Results/Supplementary Materials.